# VoxelMorph Registration & Folding Correction

End-to-end example: train a small [VoxelMorph](https://github.com/voxelmorph/voxelmorph)
registration network on synthetic 2D images, extract the predicted displacement
field, and correct any folding (negative Jacobian determinants) with the iterative
SLSQP method.

**Pipeline:**
1. Generate a synthetic dataset of 2D images (random circles/ellipses)
2. Build and train a `VxmPairwise` model (unsupervised, MSE + smoothness loss)
3. Register a held-out test pair and extract the displacement field
4. Convert to project convention `(3, 1, H, W)` with `[dz=0, dy, dx]`
5. Check for negative Jacobian determinants
6. Correct folding with `iterative_parallel`
7. Visualise before/after grids

## Install VoxelMorph

Run `setup-venv.bat` from the project root to create a `.venv` with all
dependencies (including GPU-accelerated PyTorch and VoxelMorph).  Then
select the `.venv` kernel in VS Code before running the notebook.

If you've already set up the venv, skip straight to the imports cell.

In [1]:
# Verify the environment — run this cell to check everything is importable.
import importlib, sys
print(f'Python: {sys.executable}')
for pkg in ['numpy', 'scipy', 'matplotlib', 'SimpleITK', 'nibabel', 'pandas',
            'torch', 'voxelmorph', 'neurite']:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'  {pkg:15s} {v}')
    except ImportError:
        print(f'  {pkg:15s} *** MISSING — run setup-venv.bat ***')

Python: c:\Users\Andy\Documents\GitHub\UCI-iGravi\deformation-field-processing\.venv\Scripts\python.exe
  numpy           2.3.5
  scipy           1.17.1
  matplotlib      3.10.8
  SimpleITK       2.5.3
  nibabel         5.4.2


  pandas          3.0.1
  torch           2.10.0+cu128
  voxelmorph      0.2
  neurite         0.2


## Imports

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

os.environ['NEURITE_BACKEND'] = 'pytorch'
os.environ['VXM_BACKEND'] = 'pytorch'

import neurite as ne
import voxelmorph as vxm

from dvfopt import jacobian_det2D, iterative_parallel
from dvfopt.viz import (
    plot_grid_before_after,
    plot_checkerboard_before_after,
    plot_neg_jdet_neighborhoods,
)

## Configuration

In [3]:
# --- Image / dataset ---
IMAGE_SIZE = 64          # H = W for the synthetic images
NUM_TRAIN_IMAGES = 200   # number of synthetic images for training
NUM_TEST_PAIRS = 4       # held-out pairs to register after training

# --- Model ---
NB_FEATURES = [16, 32, 32, 32, 16]   # UNet encoder/decoder feature counts
INTEGRATION_STEPS = 7                 # 0 = direct regression, >0 = diffeomorphic

# --- Training ---
EPOCHS = 200
STEPS_PER_EPOCH = 50
LR = 1e-3
LAMBDA_SMOOTH = 0.05     # weight for the spatial-gradient (smoothness) loss

# --- Correction ---
JDET_THRESHOLD = 0.01

# --- Output ---
OUTPUT_DIR = '../output/voxelmorph'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## Generate synthetic 2D dataset

Each image contains 1–4 randomly placed circles/ellipses with varying
intensities on a dark background.  This gives the network enough variety
to learn a general displacement field.

In [ ]:
def make_random_image(size, rng):
    """Generate a single synthetic 2D image with random ellipses."""
    img = np.zeros((size, size), dtype=np.float32)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)

    n_shapes = rng.integers(1, 5)
    for _ in range(n_shapes):
        cy = rng.uniform(size * 0.15, size * 0.85)
        cx = rng.uniform(size * 0.15, size * 0.85)
        ry = rng.uniform(size * 0.06, size * 0.25)
        rx = rng.uniform(size * 0.06, size * 0.25)
        intensity = rng.uniform(0.3, 1.0)
        mask = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1.0
        img[mask] = intensity

    return img


rng = np.random.default_rng(42)
n_total = NUM_TRAIN_IMAGES + NUM_TEST_PAIRS * 2  # test needs pairs
images = np.stack([make_random_image(IMAGE_SIZE, rng) for _ in range(n_total)])
train_images = images[:NUM_TRAIN_IMAGES]
test_images = images[NUM_TRAIN_IMAGES:]

print(f'Training images : {train_images.shape}')
print(f'Test images     : {test_images.shape}')

# Quick preview
fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'#{i}')
    ax.axis('off')
fig.suptitle('Sample training images')
plt.tight_layout()
plt.show()

## Build VoxelMorph model

In [ ]:
model = vxm.nn.models.VxmPairwise(
    ndim=2,
    source_channels=1,
    target_channels=1,
    nb_features=NB_FEATURES,
    integration_steps=INTEGRATION_STEPS,
    device=DEVICE,
).to(DEVICE)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Train

Unsupervised scan-to-scan training: each step draws a random pair of
images.  Losses:
- **Image loss** (MSE): reconstructed target should match the actual target
- **Smoothness loss** (spatial gradient L2): keeps the displacement field smooth

In [ ]:
image_loss_fn = ne.nn.modules.MSE()
grad_loss_fn = ne.nn.modules.SpatialGradient('l2')
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_tensor = torch.from_numpy(train_images).float().unsqueeze(1).to(DEVICE)  # (N,1,H,W)
n_train = train_tensor.shape[0]

epoch_losses = []
t0 = time.perf_counter()

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for _ in range(STEPS_PER_EPOCH):
        # Random pair
        idx = torch.randint(0, n_train, (2,))
        source = train_tensor[idx[0]].unsqueeze(0)  # (1,1,H,W)
        target = train_tensor[idx[1]].unsqueeze(0)

        optimizer.zero_grad()
        displacement, warped = model(
            source, target,
            return_warped_source=True,
            return_field_type='displacement',
        )

        img_loss = image_loss_fn(target, warped)
        smooth_loss = grad_loss_fn(displacement)
        loss = img_loss + LAMBDA_SMOOTH * smooth_loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / STEPS_PER_EPOCH
    epoch_losses.append(avg)
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:>4d}/{EPOCHS}  loss={avg:.6f}')

elapsed = time.perf_counter() - t0
print(f'\nTraining completed in {elapsed:.1f}s')

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(epoch_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.tight_layout()
plt.show()

## Register test pairs & extract displacement fields

For each test pair, run the trained model to get:
- The **warped source** image
- The **displacement field** `(1, 2, H, W)` → converted to project
  convention `(3, 1, H, W)` with `[dz=0, dy, dx]`

In [ ]:
model.eval()
test_results = []  # list of dicts

test_tensor = torch.from_numpy(test_images).float().unsqueeze(1).to(DEVICE)

with torch.no_grad():
    for i in range(NUM_TEST_PAIRS):
        source = test_tensor[2 * i].unsqueeze(0)
        target = test_tensor[2 * i + 1].unsqueeze(0)

        disp, warped = model(
            source, target,
            return_warped_source=True,
            return_field_type='displacement',
        )

        # disp: (1, 2, H, W) — VoxelMorph convention [dy, dx] for 2D
        disp_np = disp.cpu().numpy()[0]  # (2, H, W)

        # Convert to project convention: (3, 1, H, W) with [dz=0, dy, dx]
        H, W = disp_np.shape[1], disp_np.shape[2]
        deformation = np.zeros((3, 1, H, W), dtype=np.float64)
        deformation[1, 0] = disp_np[0]  # dy
        deformation[2, 0] = disp_np[1]  # dx

        test_results.append({
            'source': source.cpu().numpy()[0, 0],
            'target': target.cpu().numpy()[0, 0],
            'warped': warped.cpu().numpy()[0, 0],
            'deformation': deformation,
        })

print(f'Registered {len(test_results)} test pairs')

### Preview registration results

In [ ]:
fig, axes = plt.subplots(NUM_TEST_PAIRS, 3, figsize=(10, 3 * NUM_TEST_PAIRS))
if NUM_TEST_PAIRS == 1:
    axes = axes[np.newaxis, :]

for i, res in enumerate(test_results):
    for j, (img, label) in enumerate([
        (res['source'], 'Source (moving)'),
        (res['target'], 'Target (fixed)'),
        (res['warped'], 'Warped source'),
    ]):
        axes[i, j].imshow(img, cmap='gray', vmin=0, vmax=1)
        if i == 0:
            axes[i, j].set_title(label)
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(f'Pair {i}', fontsize=11)

plt.tight_layout()
plt.show()

## Check Jacobian determinants

Survey each test displacement field for negative Jacobian determinants.

In [ ]:
print(f'{"Pair":>6s}  {"Size":>8s}  {"Neg Jdet":>10s}  {"Below thr":>10s}  {"Min Jdet":>10s}')
print('-' * 52)

for i, res in enumerate(test_results):
    deformation = res['deformation']
    phi = np.stack([deformation[1, 0], deformation[2, 0]])  # (2, H, W)
    jac = jacobian_det2D(phi)

    n_neg = int((jac <= 0).sum())
    n_below = int((jac <= JDET_THRESHOLD).sum())
    min_j = float(jac.min())
    H, W = deformation.shape[-2:]

    res['n_neg'] = n_neg
    res['n_below'] = n_below
    res['min_jdet'] = min_j

    print(f'{i:>6d}  {H:>3d}x{W:<3d}  {n_neg:>10d}  {n_below:>10d}  {min_j:>10.4f}')

## Correct folding

Run `iterative_parallel` on each displacement field that has negative
Jacobian determinants.

In [ ]:
for i, res in enumerate(test_results):
    deformation = res['deformation']
    n_neg = res['n_neg']

    if n_neg == 0:
        print(f'Pair {i}: no negative Jacobians — skipping')
        res['phi_corrected'] = None
        continue

    save_dir = os.path.join(OUTPUT_DIR, f'pair_{i}')
    print(f'\nPair {i}: correcting {n_neg} negative Jdet pixels ...')

    t0 = time.perf_counter()
    phi = iterative_parallel(
        deformation.copy(),
        save_path=save_dir,
        verbose=1,
        threshold=JDET_THRESHOLD,
    )
    elapsed = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    min_jdet_final = float(jac_final.min())

    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))

    print(f'  Time: {elapsed:.2f}s  |  neg Jdet: {n_neg} -> {n_neg_final}  '
          f'|  min Jdet: {res["min_jdet"]:.4f} -> {min_jdet_final:.4f}  |  L2: {l2:.4f}')

    res['phi_corrected'] = phi

## Visualise

Show deformation grids and neighbourhoods for each corrected pair.

In [ ]:
for i, res in enumerate(test_results):
    phi = res.get('phi_corrected')
    if phi is None:
        continue

    deformation = res['deformation']
    title = f'VoxelMorph pair {i}'

    plot_grid_before_after(deformation, phi, title=title)
    plot_checkerboard_before_after(deformation, phi, title=title)
    plot_neg_jdet_neighborhoods(deformation, phi, title=title)

## Summary

In [ ]:
print(f'{"Pair":>6s}  {"Neg (init)":>12s}  {"Neg (final)":>12s}  {"Min Jdet":>10s}  {"Corrected":>10s}')
print('-' * 58)

for i, res in enumerate(test_results):
    phi = res.get('phi_corrected')
    if phi is not None:
        jac = jacobian_det2D(phi)
        n_neg_f = int((jac <= 0).sum())
        min_j_f = float(jac.min())
    else:
        n_neg_f = 0
        min_j_f = res['min_jdet']

    corrected = 'yes' if phi is not None else 'no'
    print(f'{i:>6d}  {res["n_neg"]:>12d}  {n_neg_f:>12d}  {min_j_f:>10.4f}  {corrected:>10s}')

## (Optional) Increase folding for demonstration

If the trained model produced mostly smooth (fold-free) fields, lower
`LAMBDA_SMOOTH` (e.g. to `0.001`) and retrain, or set
`INTEGRATION_STEPS = 0` to disable diffeomorphic integration — this
typically yields displacement fields with more folding.